# Predictor interactions

Two predictors can act jointly on a response even when neither has a
strong marginal association with it. The `interactions` pass asks whether
a fitted target model contains a non-additive effect: can the joint effect
of a predictor pair be explained by adding their two separate effects?
The result is a model diagnostic, not a correlation between the predictors.

The pass runs at standard and deep budgets, using the predictability
screen as a resource. A finding is `dependencies/predictive`, with form
`interaction_h`, explicit inputs, and a target.

## From partial dependence to H²

For a fitted model, a partial dependence fixes the selected predictor
values and averages predictions over the sampled background values of
other predictors. The pass evaluates the joint partial dependence of
`x_j, x_k` and each predictor's single partial dependence at the same
sample points. It centers all three vectors before calculating:

```text
interaction_i = PD_jk(i) - PD_j(i) - PD_k(i)
interaction_variance = mean(interaction_i ** 2)
H2 = sum(interaction_i ** 2) / sum(PD_jk(i) ** 2)
```

The reported H² is clipped to [0, 1]. A large share can belong to a tiny
effect, so the evidence also records interaction variance relative to the
target variance. `interaction_z` divides the interaction variance by its
computed standard error across the evaluation points; it is a diagnostic
separation measure, not a calibrated hypothesis-test p-value.

## Gates and search limits

A target must have a surrogate holdout R² of at least 0.10. That surrogate
must improve holdout R² over an additive, depth-one refit by at least 0.02.
This comparison helps reject interactions manufactured by a tree
approximation to an additive surface.

At standard, the pass considers up to three targets and the six strongest
predictors per target, with 128 partial-dependence/background rows. Deep
uses up to four targets and 192 rows. A reported pair must have H² at
least 0.05, `interaction_z >= 4`, and interaction variance at least 1% of
target variance. At most five pairs are retained per target.

## A product term

The response contains `2 * x1 * x2` plus an additive `x3` contribution.
The generating pair should appear among the findings for `y`.


In [ ]:
import numpy as np
import polars as pl
import onceover
from great_tables import GT, loc, style

rng = np.random.default_rng(3)
n = 1_500
x1, x2, x3 = [rng.uniform(-1, 1, n) for _ in range(3)]
y = 2 * x1 * x2 + 0.3 * x3 + rng.normal(0, 0.1, n)
df = pl.DataFrame({"x1": x1, "x2": x2, "x3": x3, "y": y})
report = onceover.profile(df, detectors=["interactions"])
findings = [f for f in report.select(category="dependencies", target="y")
            if f["specification"].get("form") == "interaction_h"]
assert any(set(f["subject"]["inputs"]) == {"x1", "x2"} for f in findings)
rows = []
for finding in findings:
    m = finding["evidence"][0]["metrics"]
    rows.append({"Inputs": ", ".join(finding["subject"]["inputs"]),
                 "H²": m["h2"], "Full-model holdout R²": m["holdout_r2"],
                 "Additive holdout R²": m["additive_holdout_r2"],
                 "Claim": finding["claim"]})
(GT(pl.DataFrame(rows))
 .fmt_number(columns=["H²", "Full-model holdout R²", "Additive holdout R²"], decimals=3)
 .tab_options(table_width="100%", table_layout="fixed", table_font_size="14px")
 .tab_style(style.css("white-space: normal; overflow-wrap: anywhere; line-height: 1.5;"), loc.body()))


In [ ]:
import altair as alt

# This displays the known generating surface, not an exported fitted model.
axis = np.linspace(-1, 1, 41)
surface = pl.DataFrame([{"x1": float(a), "x2": float(b), "effect": float(2 * a * b)}
                        for a in axis for b in axis])
(alt.Chart(surface).mark_rect()
 .encode(x=alt.X("x1:O", axis=alt.Axis(values=[-1, 0, 1]), title="x1"),
         y=alt.Y("x2:O", sort="descending", axis=alt.Axis(values=[-1, 0, 1]), title="x2"),
         color=alt.Color("effect:Q", scale=alt.Scale(scheme="redblue", domainMid=0),
                         title="2 × x1 × x2"),
         tooltip=["x1:Q", "x2:Q", "effect:Q"])
 .properties(title="The effect of x1 reverses with the sign of x2", width=390, height=280))


## Interpreting the result

The finding describes the surrogate's fitted function. Check the full and
additive holdout scores before interpreting H². Correlated predictors can
make partial-dependence combinations poorly supported by observed data;
a substantial modeled interaction does not establish a causal mechanism.

Targets and predictor pairs are shortlisted. A relationship visible only
through an interaction may be missed before this pass runs, especially if
the initial model cannot learn it or its inputs miss the shortlist.
Unreported pairs have not been shown to be additive. For model-free pair
summaries use [dependence measures](dependence_measures.ipynb); for a
single predictor's shape use [target shapes](target_shapes.ipynb).


## Method reference

The H statistic follows Friedman and Popescu's
[Predictive learning via rule ensembles (2008)](https://arxiv.org/abs/0811.1679).
Onceover applies it to its own bounded gradient-boosting surrogate and adds
the additive-refit and effect-size gates described above; it does not fit
the paper's RuleFit model as part of this pass.
